In [1]:
import logging
import os
import sys

import numpy as np
import tensorflow as tf # type: ignore

from meridian.model import model
from meridian.model import spec
from meridian.analysis import optimizer
from meridian.analysis import analyzer

from meridian.planner import flex_budget_planner
from meridian.planner.flex_budget_planner import FlexibleBudgetPlanner

In [2]:
tmp_dir = '/Users/mariappan.subramanian/Library/CloudStorage/OneDrive-TheTradeDesk/MMM/BudgetOptimizer/trash'

In [3]:
# Configure logging
logging.basicConfig(level=logging.ERROR, format='%(levelname)s: %(message)s')

# Excel file path (update this to match the actual location)
excel_file_path = (
  '/Users/mariappan.subramanian/Library/CloudStorage/'
  'OneDrive-TheTradeDesk/MMM/BudgetOptimizer/optimizer_input_case_coeff.xlsx'
)

# sample files
# 1. optimizer_input_case_coeff.xlsx --> geo + cf + media + rf inputs
# 2. optimizer_input_case_coeff_media_only.xlsx --> geo + cf + media inputs
# 3. optimizer_input_case_coeff_rf_only.xlsx --> geo + cf + rf inputs


# Check if file exists
if not os.path.exists(excel_file_path):
  print(f"ERROR: Excel file not found at {excel_file_path}")
  print("Please update the excel_file_path variable to point to the correct location.")
  sys.exit(1)

# Model configuration based on actual Excel file structure
model_config = {

  # time and geo inputs
  'time_col': 'week',
  'geo_col': 'geo',  # assumed to be national model if not given
  'population_col': 'population', # mandatory if geo_col is given

  # kpi inputs
  'kpi_col': 'conversions',  #
  'kpi_type': 'non_revenue',
  'revenue_per_kpi_col': 'revenue_per_conversion',  # needed if kpi_type is non_revenue

  # impression based media inputs
  'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
  'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
  'media_channels': ['Channel0', 'Channel1', 'Channel2'],

  # reach based media inputs
  'reach_cols': ['Channel3_reach'],
  'frequency_cols': ['Channel3_frequency'],
  'rf_spend_cols': ['Channel3_spend'],
  'rf_channels': ['Channel3'],

  }

optimization_config = {
'use_optimal_frequency': True
}

In [25]:
# base case
planner = FlexibleBudgetPlanner(file_name=excel_file_path, model_config=model_config)
opt_results = planner.optimize(optimization_config)
optimized_data = opt_results.optimized_data.sel(metric='mean')
optimized_spend = optimized_data.spend.values
optimized_outcome = optimized_data.incremental_outcome.values
print(f"{optimized_spend}")
print(f"{optimized_outcome}")


[28300000 27100000 30000000 25500000]
[5.1791016e+07 6.7949560e+07 8.7733456e+07 1.3826742e+08]


In [29]:
excel_file_path = (
  '/Users/mariappan.subramanian/Library/CloudStorage/'
  'OneDrive-TheTradeDesk/MMM/BudgetOptimizer/optimizer_input_case_roi.xlsx'
)

planner = FlexibleBudgetPlanner(file_name=excel_file_path, model_config=model_config)
opt_results = planner.optimize(optimization_config)
optimized_data = opt_results.optimized_data.sel(metric='mean')
optimized_spend = optimized_data.spend.values
optimized_outcome = optimized_data.incremental_outcome.values
print(f"{optimized_spend}")
print(f"{optimized_outcome}")


[28300000 27200000 29900000 25500000]
[5.2141900e+07 6.8567472e+07 8.7739000e+07 1.3883347e+08]


In [30]:
expected_spend = np.array([28300000, 27100000, 30000000, 25500000])
expected_outcome = np.array([5.1791016e+07, 6.7949560e+07, 8.7733456e+07, 1.3826742e+08])

In [ ]:
print(f"{optimized_spend / expected_spend}")
print(f"{optimized_outcome / expected_outcome}")


[1.         1.00369004 0.99666667 1.        ]
[1.006775   1.00909369 1.00006319 1.00409389]


In [23]:
# base case
test_excel_file_path = (
  '/Users/mariappan.subramanian/Library/CloudStorage/'
  'OneDrive-TheTradeDesk/MMM/BudgetOptimizer/model_to_excel_converter_results/demo_model_geo_all_channels_new.xlsx'
)
test_planner = FlexibleBudgetPlanner(file_name=test_excel_file_path, model_config=model_config)
test_opt_results = test_planner.optimize(optimization_config)
test_optimized_data = test_opt_results.optimized_data.sel(metric='mean')
print(f"{test_optimized_data.spend.values}")
print(f"{test_optimized_data.incremental_outcome.values}")


[28300000 27100000 30000000 25500000]
[5.1451672e+07 6.7382712e+07 8.7305000e+07 1.3812066e+08]


In [6]:
expected_spend = np.array([28300000, 27100000, 30000000, 25500000])
expected_outcome = np.array([5.1791016e+07, 6.7949560e+07, 8.7733456e+07, 1.3826742e+08])

test_spend = np.array([28300000, 27100000, 30000000, 25500000])
test_outcome = np.array([5.1451672e+07,6.7382712e+07,8.7305000e+07,1.3812066e+08])

print(f"{test_spend / expected_spend}")
print(f"{test_outcome / expected_outcome}")


[1. 1. 1. 1.]
[0.99344782 0.99165781 0.99511639 0.99893858]


In [7]:
# base case
test_excel_file_path = (
  '/Users/mariappan.subramanian/Library/CloudStorage/'
  'OneDrive-TheTradeDesk/MMM/BudgetOptimizer/model_to_excel_converter_results/demo_model_national_all_channels.xlsx'
)
test_planner = FlexibleBudgetPlanner(file_name=test_excel_file_path, model_config=model_config)
test_opt_results = test_planner.optimize(optimization_config)
test_optimized_data = test_opt_results.optimized_data.sel(metric='mean')
print(f"{test_optimized_data.spend.values}")
print(f"{test_optimized_data.incremental_outcome.values}")


/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/data/input_data_builder.py:722: UserWarning: The `population` argument is ignored in a nationally aggregated model. It will be reset to [1, 1, ..., 1]
  warnings.warn(
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/data/input_data_builder.py:722: UserWarning: The `population` argument is ignored in a nationally aggregated model. It will be reset to [1, 1, ..., 1]
  warnings.warn(
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/model.py:67: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/model.py:67: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/prior_distribution.py:1146: UserWarning: Hi

/Users/mariappan.subramanian/Documents/repo/forked/meridian/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/Users/mariappan.subramanian/Documents/repo/forked/meridian/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


[35500000 33000000 16900000 25500000]
[73164872. 79753856. 34040104. 90707088.]


In [17]:
expected_spend_nat = np.array([35500000, 33000000, 16900000, 25500000])
expected_outcome_nat = np.array([73164872., 79753856., 34040104., 90707088.])

In [21]:
saved_opt_path = '/Users/mariappan.subramanian/Documents/repo/forked/meridian/demo/saved_opts/demo_model_national_all_channels.pkl'
opt = model.load_mmm(saved_opt_path)
opt_results = opt.optimized_data.sel(metric='mean')
optimized_spend = opt_results.spend.values
optimized_outcome = opt_results.incremental_outcome.values
print(f"{optimized_spend}")
print(f"{optimized_outcome}")


[36700000 31400000 17300000 25500000]
[92422344. 88449384. 42862512. 91044728.]


In [22]:
print(f"{optimized_spend / expected_spend_nat}")
print(f"{optimized_outcome / expected_outcome_nat}")


[1.03380282 0.95151515 1.02366864 1.        ]
[1.26320653 1.10902956 1.25917688 1.00372231]


In [26]:
# Access intermediate outputs
input_data = planner.input_data        # Meridian InputData object
inference_data = planner.inference_data  # ArviZ InferenceData object
model = planner.model_obj              # Meridian model object

In [8]:
input_data.media

In [10]:
model.media_tensors.media

In [11]:
model.rf_tensors.frequency

In [11]:
19630000 / 25500000, 98957080 / 1.3826742e+08

(0.7698039215686274, 0.7156934005132952)

In [20]:
expected_optimal_spend = np.array([28300000, 27100000, 30000000, 25500000])
expected_optimal_incremental_outcome = np.array([5.1791016e+07, 6.7949560e+07, 8.7733456e+07, 1.3826742e+08])

In [23]:
print(np.array([28300000, 27100000, 30000000, 25500000]) / expected_optimal_spend)
print(np.array([5.1791016e+07, 6.7949560e+07, 8.7733456e+07, 1.2854843e+08]) / expected_optimal_incremental_outcome)

[1. 1. 1. 1.]
[1.         1.         1.         0.92970875]


In [21]:
# [28300000 27100000 30000000 25500000]
# [5.1791016e+07 6.7949560e+07 8.7733456e+07 1.3826742e+08]

In [22]:
planner = FlexibleBudgetPlanner(file_name=excel_file_path, model_config=model_config)

data = planner.build_input_data()
point_inference_data = planner.get_inference_data()

# Create Meridian model
logging.info("Creating Meridian model...")
model_spec = spec.ModelSpec()
model_obj = model.Meridian(
    input_data=data,
    model_spec=model_spec,
    inference_data=point_inference_data
)

# Sample prior (required for optimization)
logging.info("Sampling prior distributions...")
model_obj.sample_prior(n_draws=100, seed=42)


AttributeError: 'Meridian' object has no attribute 'Meridian'

In [ ]:
# get historical spend
hist_spend_media = data.media_spend.sum(dim=('geo', 'time')).data
hist_spend_rf = data.rf_spend.sum(dim=('geo', 'time')).data
hist_spend_by_channel = np.round(np.concat((hist_spend_media, hist_spend_rf)))
hist_spend_total = hist_spend_by_channel.sum()
pct_spend_by_channel = hist_spend_by_channel / hist_spend_total

In [ ]:
spend_constraint_lower = 0.7 * hist_spend_by_channel
spend_constraint_upper = 1.3 * hist_spend_by_channel
step_size = 10 ** 6
n_media_channels = model_obj.n_media_channels
n_rf_channels = model_obj.n_rf_channels
grid_rows = int(max((spend_constraint_upper - spend_constraint_lower)) // step_size + 1)
grid_cols = n_media_channels + n_rf_channels

# create spend grid
spend_grid = np.zeros((grid_rows, grid_cols))
spend_grid[0, :] = spend_constraint_lower
for i in range(1, grid_rows):
  for j in range(grid_cols):
    spend_grid[i, j] = min(spend_grid[i-1, j] + step_size, spend_constraint_upper[j])

# multiplier grid
multiplier_grid = spend_grid / hist_spend_by_channel[None, :]


In [ ]:
# get historical media units
media_scaled_tensor = model_obj.media_tensors.media_scaled
reach_scaled_tensor = model_obj.rf_tensors.reach_scaled
frequency_tensor = model_obj.rf_tensors.frequency
population = model_obj.population
population_scaled_stdev = model_obj.kpi_transformer.population_scaled_stdev
revenue_per_kpi = model_obj.revenue_per_kpi

media_idx = slice(0, n_media_channels)
rf_idx = slice(n_media_channels, n_media_channels + n_rf_channels)

In [ ]:
# get the media parameters
# 1. media
alpha_m =model_obj.inference_data.posterior.alpha_m.data
ec_m = model_obj.inference_data.posterior.ec_m.data
slope_m = model_obj.inference_data.posterior.slope_m.data
beta_gm = tf.constant(model_obj.inference_data.posterior.beta_gm.data)

# 2. rf
alpha_rf = model_obj.inference_data.posterior.alpha_rf.data
ec_rf = model_obj.inference_data.posterior.ec_rf.data
slope_rf = model_obj.inference_data.posterior.slope_rf.data
beta_grf = tf.constant(model_obj.inference_data.posterior.beta_grf.data)

In [ ]:
inc_grid = np.zeros((grid_rows, grid_cols))

for i in range(grid_rows):
  scaling_factor_array = multiplier_grid[i]

  # ------------ Media ------------
  # scale the media units
  media_scaled_tensor_i = media_scaled_tensor * scaling_factor_array[media_idx]

  # transform the media units
  media_transformed_tensor_i = model_obj.adstock_hill_media(
    media=media_scaled_tensor_i,
    alpha=alpha_m,
    ec=ec_m,
    slope=slope_m
  )

  # calculate the incremental outcome
  media_inc_outcome_tensor_i = tf.einsum("...gtx, ...gt, ...gx, ...g, ... -> ...gtx", media_transformed_tensor_i, revenue_per_kpi, beta_gm, population, population_scaled_stdev)
  media_inc_total_i = tf.reduce_sum(media_inc_outcome_tensor_i, axis=tuple(range(media_inc_outcome_tensor_i.ndim - 1))).numpy()
  inc_grid[i, media_idx] = media_inc_total_i

  # ------------ RF ------------
  reach_scaled_tensor_i = reach_scaled_tensor * scaling_factor_array[rf_idx]
  frequency_tensor_i = frequency_tensor
  rf_transformed_tensor_i = model_obj.adstock_hill_rf(
      reach=reach_scaled_tensor_i,
      frequency=frequency_tensor_i,
      alpha=alpha_rf,
      ec=ec_rf,
      slope=slope_rf
  )

  # calculate the incremental outcome
  rf_inc_outcome_tensor_i = tf.einsum("...gtx, ...gt, ...gx, ...g, ... -> ...gtx", rf_transformed_tensor_i, revenue_per_kpi, beta_grf, population, population_scaled_stdev)
  rf_inc_total_i = tf.reduce_sum(rf_inc_outcome_tensor_i, axis=tuple(range(rf_inc_outcome_tensor_i.ndim - 1))).numpy()
  inc_grid[i, rf_idx] = rf_inc_total_i

In [ ]:
inc_grid


array([[5.17787560e+07, 5.56136560e+07, 6.29544800e+07, 6.92723120e+07],
       [5.30193520e+07, 5.73952880e+07, 6.52657160e+07, 7.43134240e+07],
       [5.42330360e+07, 5.91127160e+07, 6.74761840e+07, 7.93545440e+07],
       [5.54207000e+07, 6.07699280e+07, 6.95926160e+07, 8.43956480e+07],
       [5.65833160e+07, 6.23698160e+07, 7.16215440e+07, 8.94367680e+07],
       [5.77216640e+07, 6.39157520e+07, 7.35683920e+07, 9.44778880e+07],
       [5.88364720e+07, 6.54101640e+07, 7.54385360e+07, 9.95189920e+07],
       [5.99283880e+07, 6.68559640e+07, 7.72365840e+07, 1.04560112e+08],
       [6.09981000e+07, 6.82554800e+07, 7.89669920e+07, 1.09601232e+08],
       [6.20464520e+07, 6.96109760e+07, 8.06334320e+07, 1.14642336e+08],
       [6.30740840e+07, 7.09244720e+07, 8.22398000e+07, 1.19683440e+08],
       [6.40815840e+07, 7.21982160e+07, 8.37896000e+07, 1.24724568e+08],
       [6.50693600e+07, 7.34337200e+07, 8.52857680e+07, 1.28648576e+08],
       [6.60381920e+07, 7.46330480e+07, 8.67314560e

In [ ]:
spend_grid[0] / opt_results.optimization_grid.spend_grid.data[0]

array([0.99965813, 1.00110576, 0.99914985, 1.00302704])

In [ ]:
inc_grid[0] / opt_results.optimization_grid.incremental_outcome_grid.data[0]

array([0.99976328, 1.00069693, 0.9994804 , 1.00302674])

In [ ]:
revenue_per_kpi

<tf.Tensor: shape=(20, 156), dtype=float32, numpy=
array([[0.03521255, 0.03502608, 0.03452934, ..., 0.03524379, 0.03494015,
        0.0353736 ],
       [0.0354925 , 0.03522805, 0.03525517, ..., 0.03532748, 0.03523039,
        0.03505754],
       [0.03509424, 0.03495229, 0.03470902, ..., 0.03521465, 0.03540361,
        0.03525032],
       ...,
       [0.03460286, 0.0350171 , 0.03480186, ..., 0.03482936, 0.03457915,
        0.03540254],
       [0.0347385 , 0.03499772, 0.03526273, ..., 0.03469292, 0.03470732,
        0.03496621],
       [0.03453173, 0.03521159, 0.03549562, ..., 0.03468844, 0.03472115,
        0.03523307]], dtype=float32)>

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

In [ ]:
# base case
planner = FlexibleBudgetPlanner(file_name=excel_file_path, model_config=model_config)
opt_results = planner.optimize()
optimized_data = opt_results.optimized_data.sel(metric='mean')
print(f"{optimized_data.spend.values}")
print(f"{optimized_data.incremental_outcome.values}")


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-09-09 12:45:35.727238: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


[28300000 27100000 30000000 25500000]
[5.1791016e+07 6.7949560e+07 8.7733456e+07 1.3826742e+08]


In [ ]:
# optimize kpi
config2 = model_config.copy()
config2['revenue_per_kpi_col'] = None
planner = FlexibleBudgetPlanner(file_name=excel_file_path, model_config=config2)
opt_results = planner.optimize()
optimized_data = opt_results.optimized_data.sel(metric='mean')
print(f"{optimized_data.spend.values}")
print(f"{optimized_data.incremental_outcome.values}")

/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/data/input_data.py:471: UserWarning: Consider setting custom priors, as kpi_type was specified as `non_revenue` with no `revenue_per_kpi` being set. Otherwise, the total media contribution prior will be used with `p_mean=0.4` and `p_sd=0.2`. Further documentation available at https://developers.google.com/meridian/docs/advanced-modeling/unknown-revenue-kpi-custom#set-total-paid-media-contribution-prior
  warnings.warn(
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/data/input_data.py:471: UserWarning: Consider setting custom priors, as kpi_type was specified as `non_revenue` with no `revenue_per_kpi` being set. Otherwise, the total media contribution prior will be used with `p_mean=0.4` and `p_sd=0.2`. Further documentation available at https://developers.google.com/meridian/docs/advanced-modeling/unknown-revenue-kpi-custom#set-total-paid-media-contribution-prior
  warnings.warn(
/Users/mariappan

[28300000 27100000 30000000 25500000]
[1.4799639e+09 1.9416799e+09 2.5069683e+09 3.9511265e+09]


In [ ]:
# [28300000 27100000 30000000 25500000]
# [5.1791016e+07 6.7949560e+07 8.7733456e+07 1.3826742e+08]

1. Create InputData

In [ ]:
flexible_budget_planner = FlexibleBudgetPlanner(file_name=excel_file_path, model_config=model_config)
data = flexible_budget_planner.build_input_data()

In [ ]:
flexible_budget_planner.model_config

------------- Back calculate the coefficients from ROI

In [ ]:
parameter_arrays = flexible_budget_planner.get_processed_parameter_arrays()

In [ ]:
dummy_model = model.Meridian(input_data=data, model_spec=spec.ModelSpec())


# get the arguments
mmm = dummy_model
media_tensors = mmm.media_tensors
rf_tensors = mmm.rf_tensors

alpha_m = parameter_arrays.get('alpha_m')
ec_m = parameter_arrays.get('ec_m')
slope_m = parameter_arrays.get('slope_m')

if media_tensors.media is not None:
  media_transformed = mmm.adstock_hill_media(
      media=media_tensors.media_scaled,
      alpha=tf.convert_to_tensor(alpha_m, dtype=tf.float32),
      ec=tf.convert_to_tensor(ec_m, dtype=tf.float32),
      slope=tf.convert_to_tensor(slope_m, dtype=tf.float32),
  )

In [ ]:
media_transformed

<tf.Tensor: shape=(20, 156, 3), dtype=float32, numpy=
array([[[0.2908246 , 0.00296332, 0.44369364],
        [0.3267742 , 0.3654256 , 0.50572467],
        [0.38526192, 0.2995557 , 0.526262  ],
        ...,
        [0.41003463, 0.54106367, 0.15925233],
        [0.37518638, 0.3674578 , 0.34121776],
        [0.4047346 , 0.5397538 , 0.40995178]],

       [[0.31641188, 0.33694944, 0.32015955],
        [0.39815843, 0.4397392 , 0.46710268],
        [0.43406695, 0.48160437, 0.43281072],
        ...,
        [0.4123682 , 0.52594423, 0.17299233],
        [0.32331753, 0.24132252, 0.47541517],
        [0.35831326, 0.55271375, 0.5133842 ]],

       [[0.27430627, 0.43054658, 0.26169476],
        [0.3479516 , 0.45581812, 0.2675743 ],
        [0.40277955, 0.49288186, 0.474427  ],
        ...,
        [0.40987837, 0.54559594, 0.38836136],
        [0.35093376, 0.44849288, 0.4184708 ],
        [0.37151918, 0.5361595 , 0.37130007]],

       ...,

       [[0.2974166 , 0.09182433, 0.09301934],
        [0.372

In [ ]:
data.media

<xarray.DataArray 'media' (geo: 20, media_time: 156, media_channel: 3)> Size: 75kB
array([[[1392518,    3733,  670235],
        [ 937228,  722210,  745025],
        [1286569,  329778,  786262],
        ...,
        [1211774, 1173873,       0],
        [ 836566,  305098,  407998],
        [1269842, 1263794,  509309]],

       [[3032312, 1231404,  763501],
        [2785805, 1548845, 1290322],
        [2811866, 1705897,  993765],
        ...,
        [2274402, 2511346,   14731],
        [ 786126,       0, 1411209],
        [2067059, 2773346, 1458800]],

       [[ 593030,  438756,  137622],
        [ 534426,  360286,  118274],
        [ 630650,  424657,  326189],
        ...,
...
        ...,
        [ 249370,  264544,   21748],
        [ 183341,   49802,  176608],
        [ 275637,  306384,  106971]],

       [[2582698,  922861,  653117],
        [1424830, 1275632,  962646],
        [2497179, 1203092, 1632313],
        ...,
        [2530296, 1230214,       0],
        [ 946695,  351198, 1537708],
        [2415999, 2758301,  647724]],

       [[2304498,  753260,  255735],
        [2036626, 1433122, 1656609],
        [2946513, 1484241, 1975205],
        ...,
        [2387086, 1703876,       0],
        [ 944313, 1320312, 1500085],
        [2571754, 2509916, 1247641]]])
Coordinates:
  * media_channel  (media_channel) object 24B 'Channel0' 'Channel1' 'Channel2'
  * media_time     (media_time) <U10 6kB '2021-01-25' ... '2024-01-15'
  * geo            (geo) <U5 400B 'Geo0' 'Geo1' 'Geo2' ... 'Geo18' 'Geo19'

In [ ]:
parameter_arrays = flexible_budget_planner.get_processed_parameter_arrays()
coefficient_arrays = flexible_budget_planner.get_processed_coefficients_arrays()

In [ ]:
parameter_arrays

{'alpha_m': <xarray.DataArray 'alpha_m' (media_channel: 3)> Size: 24B
 array([0.51056719, 0.28670812, 0.17126298])
 Coordinates:
   * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2',
 'ec_m': <xarray.DataArray 'ec_m' (media_channel: 3)> Size: 24B
 array([1.52952576, 1.23229408, 1.16474795])
 Coordinates:
   * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2',
 'slope_m': <xarray.DataArray 'slope_m' (media_channel: 3)> Size: 24B
 array([1., 1., 1.])
 Coordinates:
   * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2',
 'alpha_rf': <xarray.DataArray 'alpha_rf' (rf_channel: 1)> Size: 8B
 array([0.47872618])
 Coordinates:
   * rf_channel  (rf_channel) <U8 32B 'Channel3',
 'ec_rf': <xarray.DataArray 'ec_rf' (rf_channel: 1)> Size: 8B
 array([1.25419044])
 Coordinates:
   * rf_channel  (rf_channel) <U8 32B 'Channel3',
 'slope_rf': <xarray.DataArray 'slope_rf' (rf_channel: 1)> Size: 8B
 array([3.89582825])
 Coordinates:
   * r

In [ ]:
parameter_arrays['alpha_m']

<xarray.DataArray 'alpha_m' (media_channel: 3)> Size: 24B
array([0.51056719, 0.28670812, 0.17126298])
Coordinates:
  * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'

In [ ]:
coefficient_arrays

{'beta_gm': <xarray.DataArray 'beta_gm' (geo: 20, media_channel: 3)> Size: 480B
 array([[0.59156996, 0.57695675, 0.65051496],
        [0.60544699, 0.58618772, 0.61531818],
        [0.5801062 , 0.553298  , 0.6111849 ],
        [0.57807249, 0.55831456, 0.62035668],
        [0.57906914, 0.55886364, 0.62508488],
        [0.57296002, 0.56320864, 0.6191318 ],
        [0.57095027, 0.55212671, 0.6290918 ],
        [0.58986926, 0.56343049, 0.64753032],
        [0.57893503, 0.58381557, 0.62021452],
        [0.5705474 , 0.54913819, 0.63340759],
        [0.61041367, 0.54180527, 0.64330077],
        [0.58996964, 0.56087613, 0.64733016],
        [0.62054336, 0.52024597, 0.63420427],
        [0.5711937 , 0.56435418, 0.64072132],
        [0.54389858, 0.55787253, 0.6380682 ],
        [0.58158261, 0.56385761, 0.63503915],
        [0.63654286, 0.54719704, 0.62848502],
        [0.59715307, 0.55126083, 0.63297832],
        [0.59599978, 0.55974191, 0.6346727 ],
        [0.56065822, 0.51651025, 0.61709756]])

2. Create InferenceData based on the point estimates in the input excel

In [ ]:
inference_data = flexible_budget_planner.get_inference_data()

3. Create MMM object and add InferenceData to it

In [ ]:
model_spec = spec.ModelSpec()
mmm = model.Meridian(input_data=data, model_spec=model_spec, inference_data=inference_data)
mmm.sample_prior(n_draws=100, seed=42)  # does not affect the optimization results


In [ ]:
mmm.inference_data.posterior

<xarray.Dataset> Size: 5kB
Dimensions:           (chain: 1, draw: 1, media_channel: 3, rf_channel: 1,
                       geo: 20, time: 156, knots: 156, control_variable: 2)
Coordinates:
  * chain             (chain) int64 8B 0
  * draw              (draw) int64 8B 0
  * media_channel     (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'
  * rf_channel        (rf_channel) <U8 32B 'Channel3'
  * geo               (geo) <U5 400B 'Geo0' 'Geo1' 'Geo10' ... 'Geo8' 'Geo9'
  * time              (time) int64 1kB 0 1 2 3 4 5 6 ... 150 151 152 153 154 155
  * knots             (knots) int64 1kB 0 1 2 3 4 5 ... 150 151 152 153 154 155
  * control_variable  (control_variable) <U33 264B 'sentiment_score_control' ...
Data variables: (12/24)
    alpha_m           (chain, draw, media_channel) float32 12B 0.5106 ... 0.1713
    ec_m              (chain, draw, media_channel) float32 12B 1.53 1.232 1.165
    slope_m           (chain, draw, media_channel) float32 12B 1.0 1.0 1.0
    alpha_rf          (chain, draw, rf_channel) float32 4B 0.4787
    ec_rf             (chain, draw, rf_channel) float32 4B 1.254
    slope_rf          (chain, draw, rf_channel) float32 4B 3.896
    ...                ...
    contribution_rf   (chain, draw, rf_channel) float32 4B 0.0
    beta_rf           (chain, draw, rf_channel) float32 4B 0.0
    eta_rf            (chain, draw, rf_channel) float32 4B 0.0
    gamma_c           (chain, draw, control_variable) float32 8B 0.0 0.0
    xi_c              (chain, draw, control_variable) float32 8B 0.0 0.0
    gamma_gc          (chain, draw, geo, control_variable) float32 160B 0.0 ....

In [ ]:
inference_data.posterior

<xarray.Dataset> Size: 5kB
Dimensions:           (chain: 1, draw: 1, media_channel: 3, rf_channel: 1,
                       geo: 20, time: 156, knots: 156, control_variable: 2)
Coordinates:
  * chain             (chain) int64 8B 0
  * draw              (draw) int64 8B 0
  * media_channel     (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'
  * rf_channel        (rf_channel) <U8 32B 'Channel3'
  * geo               (geo) <U5 400B 'Geo0' 'Geo1' 'Geo10' ... 'Geo8' 'Geo9'
  * time              (time) int64 1kB 0 1 2 3 4 5 6 ... 150 151 152 153 154 155
  * knots             (knots) int64 1kB 0 1 2 3 4 5 ... 150 151 152 153 154 155
  * control_variable  (control_variable) <U33 264B 'sentiment_score_control' ...
Data variables: (12/24)
    alpha_m           (chain, draw, media_channel) float32 12B 0.5106 ... 0.1713
    ec_m              (chain, draw, media_channel) float32 12B 1.53 1.232 1.165
    slope_m           (chain, draw, media_channel) float32 12B 1.0 1.0 1.0
    alpha_rf          (chain, draw, rf_channel) float32 4B 0.4787
    ec_rf             (chain, draw, rf_channel) float32 4B 1.254
    slope_rf          (chain, draw, rf_channel) float32 4B 3.896
    ...                ...
    contribution_rf   (chain, draw, rf_channel) float32 4B 0.0
    beta_rf           (chain, draw, rf_channel) float32 4B 0.0
    eta_rf            (chain, draw, rf_channel) float32 4B 0.0
    gamma_c           (chain, draw, control_variable) float32 8B 0.0 0.0
    xi_c              (chain, draw, control_variable) float32 8B 0.0 0.0
    gamma_gc          (chain, draw, geo, control_variable) float32 160B 0.0 ....

##### ~~~~~~~~ Contribution and ROI calculation ~~~~~~~~~~~~

In [ ]:
mmm_analyzer = analyzer.Analyzer(mmm)
inc_outcome = mmm_analyzer.incremental_outcome(use_posterior=True)
inc_outcome.shape

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-09-02 10:21:00.469581: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


TensorShape([1, 1, 4])

In [ ]:
inc_outcome

<tf.Tensor: shape=(1, 1, 4), dtype=float32, numpy=array([[[65190904., 68640128., 77200936., 98960488.]]], dtype=float32)>

In [ ]:
summary_metrics = mmm_analyzer.summary_metrics(aggregate_geos=False)
geo_summary = summary_metrics.sel(metric='mean',distribution='posterior').to_dataframe()

In [ ]:
geo_summary.reset_index().to_csv(os.path.join(tmp_dir, 'geo_roi_summary.csv'))

In [ ]:
import tensorflow as tf

# Shapes: g=2, t=5, x=3 (no batch dims)
linear_diff = tf.constant([
  # g = 0  -> 5 rows over t, 3 cols over x
  [[ 1,  2, 3],
    [ 0,  1, 1],
    [-1,  2, 0],
    [ 2,  0, 1],
    [ 1, -1, 2]],
  # g = 1
  [[ 1, -1,  0],
    [ 2,  1,  1],
    [ 0,  2,  1],
    [-1,  0,  2],
    [ 1,  1, -1]],
], dtype=tf.float32)  # shape: (2, 5, 3)

revenue_per_kpi = tf.constant([
  [10, 20, 0, 5, 5],   # g=0
  [ 1,  0, 2, 0, 1],   # g=1
], dtype=tf.float32)      # shape: (2, 5)

population = tf.constant([2, 3], dtype=tf.float32)   # shape: (2,)
population_scaled_stdev = tf.constant(0.5, tf.float32)  # scalar

# --- Einsum version ---
# "...gtx,gt,g,->...gx" with no batch dims ("...") -> "gtx,gt,g,->gx"
out_einsum = tf.einsum("gtx,gt,g,->gx",
                      linear_diff,
                      revenue_per_kpi,
                      population,
                      population_scaled_stdev)

In [ ]:
out_einsum

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[25. , 35. , 65. ],
       [ 3. ,  6. ,  1.5]], dtype=float32)>

In [ ]:
linear_diff.shape

TensorShape([2, 5, 3])

In [ ]:
expression_eval = linear_diff * revenue_per_kpi[:, :, None] * population[:, None, None] * population_scaled_stdev
tf.reduce_sum(expression_eval, axis=-2)

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[25. , 35. , 65. ],
       [ 3. ,  6. ,  1.5]], dtype=float32)>

<tf.Tensor: shape=(2, 1, 1), dtype=float32, numpy=
array([[[2.]],

       [[3.]]], dtype=float32)>

4. Call the Optimizer

<!-- %%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize(use_posterior=True) -->

In [ ]:
# optimization_results.plot_spend_delta()

In [ ]:
# optimization_results.plot_incremental_outcome_delta()

In [ ]:
# optimization_results.plot_budget_allocation()

In [ ]:
# optimization_results.plot_response_curves()

In [ ]:
# optimization_results.optimized_data

3. Let's manually run the optimization routines

In [ ]:
from collections.abc import Mapping, Sequence
import dataclasses
import functools
import math
import os
from typing import Any, TypeAlias
import warnings

import altair as alt
import jinja2
from meridian import constants as c
from meridian.analysis import analyzer
from meridian.analysis import formatter
from meridian.analysis import summary_text
from meridian.data import time_coordinates as tc
from meridian.model import model
import numpy as np
import pandas as pd
import tensorflow as tf
import xarray as xr

from meridian.analysis.optimizer import _SpendConstraint, OptimizationGrid, _validate_budget

In [ ]:
# initialize
from meridian.analysis import optimizer
self = optimizer.BudgetOptimizer(mmm)

In [ ]:
new_data: analyzer.DataTensors | None = None
use_posterior: bool = True
selected_times: tuple[str | None, str | None] | None = None
start_date: tc.Date = None
end_date: tc.Date = None
fixed_budget: bool = True
budget: float | None = None
pct_of_spend: Sequence[float] | None = None
spend_constraint_lower: _SpendConstraint | None = None
spend_constraint_upper: _SpendConstraint | None = None
target_roi: float | None = None
target_mroi: float | None = None
gtol: float = 0.0001
use_optimal_frequency: bool = True
use_kpi: bool = False
confidence_level: float = c.DEFAULT_CONFIDENCE_LEVEL
batch_size: int = c.DEFAULT_BATCH_SIZE
optimization_grid: OptimizationGrid | None = None

In [ ]:
if selected_times is not None:
  warnings.warn(
      '`selected_times` is deprecated. Please use `start_date` and'
      ' `end_date` instead.',
      DeprecationWarning,
      stacklevel=2,
  )
  deprecated_start_date, deprecated_end_date = selected_times
  start_date = start_date or deprecated_start_date
  end_date = end_date or deprecated_end_date

_validate_budget(
    fixed_budget=fixed_budget,
    budget=budget,
    target_roi=target_roi,
    target_mroi=target_mroi,
)

In [ ]:
spend_constraint_default = (
    c.SPEND_CONSTRAINT_DEFAULT_FIXED_BUDGET
    if fixed_budget
    else c.SPEND_CONSTRAINT_DEFAULT_FLEXIBLE_BUDGET
)

if spend_constraint_lower is None:
  spend_constraint_lower = spend_constraint_default
if spend_constraint_upper is None:
  spend_constraint_upper = spend_constraint_default


In [ ]:
spend_constraint_default, spend_constraint_lower, spend_constraint_upper

(0.3, 0.3, 0.3)

In [ ]:
use_grid_arg = optimization_grid is not None and self._validate_grid(
    new_data=new_data,
    use_posterior=use_posterior,
    start_date=start_date,
    end_date=end_date,
    budget=budget,
    pct_of_spend=pct_of_spend,
    spend_constraint_lower=spend_constraint_lower,
    spend_constraint_upper=spend_constraint_upper,
    gtol=gtol,
    use_optimal_frequency=use_optimal_frequency,
    use_kpi=use_kpi,
    optimization_grid=optimization_grid,
)

use_grid_arg

False

In [ ]:
# if optimization_grid is None or not use_grid_arg:
#   optimization_grid = self.create_optimization_grid(
#       new_data=new_data,
#       start_date=start_date,
#       end_date=end_date,
#       budget=budget,
#       pct_of_spend=pct_of_spend,
#       spend_constraint_lower=spend_constraint_lower,
#       spend_constraint_upper=spend_constraint_upper,
#       gtol=gtol,
#       use_posterior=use_posterior,
#       use_kpi=use_kpi,
#       use_optimal_frequency=use_optimal_frequency,
#       batch_size=batch_size,
#   )

# inside create_optimization_grid
new_data=new_data
start_date=start_date
end_date=end_date
budget=budget
pct_of_spend=pct_of_spend
spend_constraint_lower=spend_constraint_lower
spend_constraint_upper=spend_constraint_upper
gtol=gtol
use_posterior=use_posterior
use_kpi=use_kpi
use_optimal_frequency=use_optimal_frequency
batch_size=batch_size


In [ ]:
self._validate_model_fit(use_posterior)
if new_data is None:
  new_data = analyzer.DataTensors()

required_tensors = c.PERFORMANCE_DATA + (c.TIME,)
filled_data = new_data.validate_and_fill_missing_data(
    required_tensors_names=required_tensors, meridian=self._meridian
)

In [ ]:
hist_spend = self._analyzer.get_aggregated_spend(
    new_data=filled_data.filter_fields(c.PAID_CHANNELS + c.SPEND_DATA),
    selected_times=selected_times,
    include_media=self._meridian.n_media_channels > 0,
    include_rf=self._meridian.n_rf_channels > 0,
).data

In [ ]:
hist_spend

array([40414664., 27601934., 23265924., 19630668.], dtype=float32)

In [ ]:
new_data_hist = filled_data.filter_fields(c.PAID_CHANNELS + c.SPEND_DATA)
tf.reduce_sum(new_data_hist.media_spend, axis=(0, 1))

<tf.Tensor: shape=(3,), dtype=float32, numpy=array([40414664., 27601934., 23265924.], dtype=float32)>

In [ ]:
c.PAID_CHANNELS + c.SPEND_DATA

('media', 'reach', 'frequency', 'media_spend', 'rf_spend')

In [ ]:
mmm.input_data.media_channel.values

array(['Channel0', 'Channel1', 'Channel2'], dtype=object)

In [ ]:
media_spend_tensor = filled_data.media_spend
media_spend_tensor.ndim


3

In [ ]:
allowed_n_channels = [
    mmm.n_media_channels,
    mmm.n_rf_channels,
    mmm.n_media_channels + mmm.n_rf_channels,
    mmm.n_media_channels
    + mmm.n_rf_channels
    + mmm.n_non_media_channels
    + mmm.n_organic_media_channels
    + mmm.n_organic_rf_channels,
]

In [ ]:
has_media_dim = True
aggregate_geos = True
aggregate_times = True
tensor_dims = "...gt" + "m" * has_media_dim
output_dims = (
    "g" * (not aggregate_geos)
    + "t" * (not aggregate_times)
    + "m" * has_media_dim
)

In [ ]:
tensor_dims

'...gtm'

In [ ]:
output_dims

'm'

In [ ]:
tensor = media_spend_tensor
tf.einsum(f"{tensor_dims}->...{output_dims}", tensor)

<tf.Tensor: shape=(3,), dtype=float32, numpy=array([40414664., 27601934., 23265924.], dtype=float32)>

In [ ]:
f"{tensor_dims}->...{output_dims}"

'...gtm->...m'

In [ ]:
tf.reduce_sum(tensor, axis=(0, 1))

<tf.Tensor: shape=(3,), dtype=float32, numpy=array([40414664., 27601934., 23265924.], dtype=float32)>

In [ ]:
optimization_grid.grid_dataset

<xarray.Dataset> Size: 18kB
Dimensions:                   (grid_spend_index: 243, channel: 4)
Coordinates:
  * grid_spend_index          (grid_spend_index) int64 2kB 0 1 2 ... 240 241 242
  * channel                   (channel) object 32B 'Channel0' ... 'Channel3'
Data variables:
    spend_grid                (grid_spend_index, channel) float64 8kB 2.83e+0...
    incremental_outcome_grid  (grid_spend_index, channel) float64 8kB 5.179e+...
Attributes:
    spend_step_size:  100000

In [ ]:
mean_opt_results = optimization_grid.grid_dataset.mean(dim="grid_spend_index")
mean_opt_results.to_pandas()

,spend_grid,incremental_outcome_grid
channel,,
Channel0,40400000.0,6.470639e+07
Channel1,27600000.0,6.812717e+07
Channel2,23300000.0,7.669325e+07
Channel3,19600000.0,1.062761e+08


In [ ]:
# from meridian.analysis import optimizer
# budget_optimizer = optimizer.BudgetOptimizer(mmm)
# optimization_results = budget_optimizer.optimize()

In [ ]:
# param_arrays_dict = adhoc_data_loader.get_processed_parameter_arrays()
# coeff_arrays_dict = adhoc_data_loader.get_processed_coefficients_arrays()

INFO: Coefficients validation passed - all requirements satisfied
INFO: Successfully created coefficients DataArrays for 2 coefficient types
INFO: Successfully processed media coefficients as DataArrays


In [ ]:
# from meridian.model import model
# from meridian.model import spec


# # Configure the model
# model_spec = spec.ModelSpec()
# mmm = model.Meridian(input_data=data, model_spec=model_spec)

In [ ]:
from unittest import mock

from meridian.data import input_data

In [ ]:
mock_data = mock.MagicMock(input_data)
mock_data

<MagicMock spec='module' id='5820205344'>

<MagicMock name='mock.InputData' id='5820203712'>